# Modèle IA v3 — Prédiction de Carences en Vitamines

**Améliorations par rapport au v2 :**
- Ablation study formalisée (impact des features symptômes ~3%)
- Hyperparameter tuning (RandomizedSearchCV)
- RepeatedStratifiedKFold (5 folds × 3 répétitions)
- Courbes ROC par classe
- Courbes d'apprentissage (Learning Curves)


In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import pickle
import os

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.model_selection import (
    RepeatedStratifiedKFold, cross_val_score,
    train_test_split, RandomizedSearchCV, learning_curve
)
from sklearn.metrics import (
    classification_report, accuracy_score, f1_score,
    roc_curve, auc, ConfusionMatrixDisplay, confusion_matrix
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
print("Imports OK")


Imports OK


In [ ]:
# ── Chargement ──────────────────────────────────────────
csv_path = os.path.join(os.path.dirname(os.getcwd()), 
                        'data_csv', 'raw', 
                        'vitamin_deficiency_disease_dataset_20260123.ods')
df = pd.read_excel(csv_path, engine='odf')
print(f"Dataset : {df.shape[0]} lignes × {df.shape[1]} colonnes")
print(f"\nColonnes :\n{df.columns.tolist()}")

# ── Détection automatique de la colonne cible ────────────
target_candidates = ['disease', 'Disease', 'vitamin_deficiency_disease', 
                     'target', 'label', 'condition', 'Condition']
target_col = None
for col in target_candidates:
    if col in df.columns:
        target_col = col
        break

if target_col is None:
    print("\n Colonne cible introuvable — voir la liste des colonnes ci-dessus")
else:
    print(f"\n Colonne cible : '{target_col}'")
    print(f"Classes : {df[target_col].unique()}")

    # ── Suppression des colonnes non cliniques ────────────
    cols_to_drop = [
        'age', 'gender', 'bmi', 'smoking_status', 'alcohol_consumption',
        'exercise_level', 'diet_type', 'sun_exposure',
        'latitude_region', 'income_level',
        'symptoms_list', 'symptoms_count'
    ]
    cols_to_drop = [c for c in cols_to_drop if c in df.columns]
    df = df.drop(columns=cols_to_drop)

    # ── Encodage de la cible ──────────────────────────────
    le = LabelEncoder()
    y = le.fit_transform(df[target_col])
    X = df.drop(columns=[target_col])

    # ── Encodage des colonnes catégorielles ───────────────
    for col in X.select_dtypes(include='object').columns:
        X[col] = LabelEncoder().fit_transform(X[col])

    # ── Split train/test ──────────────────────────────────
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    print(f"\nFeatures conservées : {X.shape[1]}")
    print(f"Train : {X_train.shape[0]} | Test : {X_test.shape[0]}")
    print(f"\nDistribution :")
    for i, cls in enumerate(le.classes_):
        print(f"  {cls} : {(y==i).sum()}")


Dataset : 4000 lignes × 34 colonnes

Colonnes :
['age', 'gender', 'bmi', 'smoking_status', 'alcohol_consumption', 'exercise_level', 'diet_type', 'sun_exposure', 'income_level', 'latitude_region', 'vitamin_a_percent_rda', 'vitamin_c_percent_rda', 'vitamin_d_percent_rda', 'vitamin_e_percent_rda', 'vitamin_b12_percent_rda', 'folate_percent_rda', 'calcium_percent_rda', 'iron_percent_rda', 'hemoglobin_g_dl', 'serum_vitamin_d_ng_ml', 'serum_vitamin_b12_pg_ml', 'serum_folate_ng_ml', 'symptoms_count', 'symptoms_list', 'has_night_blindness', 'has_fatigue', 'has_bleeding_gums', 'has_bone_pain', 'has_muscle_weakness', 'has_numbness_tingling', 'has_memory_problems', 'has_pale_skin', 'disease_diagnosis', 'has_multiple_deficiencies']

❌ Colonne cible introuvable — voir la liste des colonnes ci-dessus
